<a href="https://colab.research.google.com/github/youngbean0519/Self-Project/blob/main/notebooks/04_tfifd_logistic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/youngbean0519/Self-Project.git

Cloning into 'Self-Project'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 58 (delta 15), reused 48 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 472.47 KiB | 8.91 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [2]:
%cd Self-Project/
%ls

/content/Self-Project
notebooks/  README.md  requirements.txt  results/  src/  test.py


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import sys

PROJECT_DIR = "/content/Self-Project"
SRC_DIR = os.path.join(PROJECT_DIR, "src")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
DATA_DIR = "/content/drive/MyDrive/26_KOTE/data"
ARTIFACT_DIR = "/content/drive/MyDrive/26_KOTE/artifacts/week4_lr"

sys.path.append(PROJECT_DIR)
sys.path.append(SRC_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("DATA_DIR:", DATA_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

PROJECT_DIR: /content/Self-Project
SRC_DIR: /content/Self-Project/src
RESULTS_DIR: /content/Self-Project/results
DATA_DIR: /content/drive/MyDrive/26_KOTE/data
ARTIFACT_DIR: /content/drive/MyDrive/26_KOTE/artifacts/week4_lr


In [5]:
import preprocessing.text as text_preprocessing
import preprocessing.labels as preprocessing_labels
import evaluation.metrics as metrics
import utils.labels as utils_labels

In [6]:
import time
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

In [7]:
train_path = os.path.join(DATA_DIR, "train.tsv")
val_path = os.path.join(DATA_DIR, "val.tsv")
test_path = os.path.join(DATA_DIR, "test.tsv")

train_df = pd.read_csv(train_path, sep="\t", header=None, names=["id", "text", "labels"])
val_df = pd.read_csv(val_path, sep="\t", header=None, names=["id", "text", "labels"])
test_df = pd.read_csv(test_path, sep="\t", header=None, names=["id", "text", "labels"])

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (40000, 3)
val: (5000, 3)
test: (5000, 3)


In [8]:
NUM_LABELS = 44

train_label_lists = train_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()
val_label_lists = val_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()
test_label_lists = test_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()

y_train, mlb = preprocessing_labels.encode_multilabels(train_label_lists, NUM_LABELS)
y_val = mlb.transform(val_label_lists)
y_test = mlb.transform(test_label_lists)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

y_train: (40000, 44)
y_val: (5000, 44)
y_test: (5000, 44)


In [9]:
def make_preprocessed_texts(df, normalize_repeat, remove_symbols):
  return df["text"].astype(str).apply(
      lambda text: text_preprocessing.preprocess_text(
          text,
          normalize_repeat=normalize_repeat,
          remove_symbols=remove_symbols
      )
  )

preprocessing_conditions = {
    "raw": {
        "normalize_repeat": False,
        "remove_symbols": False
    },

    "repeat": {
        "normalize_repeat": True,
        "remove_symbols": False
    },

    "repeat_symbols": {
        "normalize_repeat": True,
        "remove_symbols": True
    }
}

In [10]:
# 전처리 적용 결과가 실제로 얼마나 달라지는지 확인

sample = train_df["text"].astype(str)

sample_repeat = sample.apply(
    lambda x: text_preprocessing.preprocess_text(
        x,
        normalize_repeat=True,
        remove_symbols=False
    )
)

sample_repeat_symbols = sample.apply(
    lambda x: text_preprocessing.preprocess_text(
        x,
        normalize_repeat=True,
        remove_symbols=True
    )
)

print(
    "raw -> repeat changed:",
    (sample != sample_repeat).sum()
)

print(
    "repeat -> repeat_symbols changed:",
    (sample_repeat != sample_repeat_symbols).sum()
)

print(
    "raw -> repeat_symbols changed:",
    (sample != sample_repeat_symbols).sum()
)

raw -> repeat changed: 13144
repeat -> repeat_symbols changed: 25198
raw -> repeat_symbols changed: 28865


In [11]:
# 반복문자 정규화로 실제 변경된 예시 확인

changed_repeat = sample != sample_repeat

for original, processed in zip(
    sample[changed_repeat].head(20),
    sample_repeat[changed_repeat].head(20)
):
    print("ORIGINAL :", original)
    print("REPEAT   :", processed)
    print("-" * 80)

ORIGINAL : 내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.
REPEAT   : 내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
--------------------------------------------------------------------------------
ORIGINAL : 미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
REPEAT   : 미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋ
--------------------------------------------------------------------------------
ORIGINAL : 어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋㅋㅋ
REPEAT   : 어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋ
--------------------------------------------------------------------------------
ORIGINAL : 물갈이약 구매했어요...미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데... 약간 불량들이 있네요... 1) 함께 구매한 스포이드에 구멍이 있네요... 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요... 3) 예전에 구매한 온도계의 온도가 안맞아요...
REPEAT   : 물갈이약 구매했어요..미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데.. 약간 불량들이 있네요.. 1) 함께 구매한 스포이드에 구멍이 있네요.. 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요.. 3) 예전에 구매한 온도계의 온도가 안맞아요..
--------------------------------------------------------------------------------
ORIGINAL : 스토리 담당만 붙여줬으면 갓작품 됬을텐데.....
RE

In [12]:
# 선택적 기호 제거로 실제 변경된 예시 확인

changed_symbols = sample_repeat != sample_repeat_symbols

for before, after in zip(
    sample_repeat[changed_symbols].head(20),
    sample_repeat_symbols[changed_symbols].head(20)
):
    print("BEFORE :", before)
    print("AFTER  :", after)
    print("-" * 80)

BEFORE : 내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
AFTER  : 내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네
--------------------------------------------------------------------------------
BEFORE : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
AFTER  : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다
--------------------------------------------------------------------------------
BEFORE : 새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
AFTER  : 새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다 그걸 가능케 한 사랑에 경의를 표합니다
--------------------------------------------------------------------------------
BEFORE : 물갈이약 구매했어요..미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데.. 약간 불량들이 있네요.. 1) 함께 구매한 스포이드에 구멍이 있네요.. 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요.. 3) 예전에 구매한 온도계의 온도가 안맞아요..
AFTER  : 물갈이약 구매했어요 미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데 약간 불량들이 있네요 1 함께 구매한 스포이드에 구멍이 있네요 2 부화통에 같이오는 흡착고무가 불량이라 안붙어요 3 예전에 구매한 온도계의 온도가 안맞아요

In [13]:
THRESHOLDS = np.arange(0.10, 0.61, 0.05)

TFIDF_PARAMS = {
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 50000,
    "sublinear_tf": True
}


def run_lr_experiment(
    train_texts,
    val_texts,
    y_train,
    y_val
):
    vectorizer = TfidfVectorizer(**TFIDF_PARAMS)

    X_train = vectorizer.fit_transform(train_texts)
    X_val = vectorizer.transform(val_texts)

    model = OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000
        )
    )

    train_start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    train_time = time.perf_counter() - train_start

    inference_start = time.perf_counter()

    y_val_prob = model.predict_proba(
        X_val
    )

    inference_time = time.perf_counter() - inference_start

    threshold_rows = []

    for threshold in THRESHOLDS:

        y_val_pred = metrics.apply_threshold(
            y_val_prob,
            threshold=float(threshold)
        )

        result = metrics.evaluate_multilabel(
            y_val,
            y_val_pred,
            y_val_prob
        )

        threshold_rows.append({
            "threshold": round(float(threshold), 2),
            "micro_f1": result["micro_f1"],
            "macro_f1": result["macro_f1"],
            "hamming_loss": result["hamming_loss"]
        })

    threshold_df = pd.DataFrame(
        threshold_rows
    )

    probability_metrics = metrics.evaluate_multilabel(
        y_val,
        metrics.apply_threshold(y_val_prob, 0.5),
        y_val_prob
    )

    return {
        "vectorizer": vectorizer,
        "model": model,
        "y_val_prob": y_val_prob,
        "threshold_results": threshold_df,
        "pr_auc_micro": probability_metrics["pr_auc_micro"],
        "pr_auc_macro": probability_metrics["pr_auc_macro"],
        "train_time": train_time,
        "inference_time": inference_time,
        "vocab_size": len(vectorizer.vocabulary_)
    }

In [15]:
experiments = {}

for condition_name, condition in preprocessing_conditions.items():
  print(f"Running: {condition_name}")

  train_texts = make_preprocessed_texts(
      train_df,
      normalize_repeat=condition["normalize_repeat"],
      remove_symbols=condition["remove_symbols"]
  )

  val_texts = make_preprocessed_texts(
      val_df,
      normalize_repeat=condition["normalize_repeat"],
      remove_symbols=condition["remove_symbols"]
  )

  experiments[condition_name] = run_lr_experiment(
      train_texts,
      val_texts,
      y_train,
      y_val
  )

  print("Done")

Running: raw
Done
Running: repeat
Done
Running: repeat_symbols
Done


In [17]:
for condition_name, experiment in experiments.items():
    print(
        condition_name,
        experiment["vocab_size"]
    )

raw 50000
repeat 50000
repeat_symbols 50000


In [18]:
best_rows = []

for condition_name, experiment in experiments.items():

    threshold_df = experiment["threshold_results"]

    best = threshold_df.sort_values(
        by=[
            "micro_f1",
            "macro_f1",
            "hamming_loss"
        ],
        ascending=[
            False,
            False,
            True
        ]
    ).iloc[0]

    best_rows.append({
        "preprocessing": condition_name,
        "threshold": best["threshold"],
        "micro_f1": best["micro_f1"],
        "macro_f1": best["macro_f1"],
        "hamming_loss": best["hamming_loss"],
        "pr_auc_micro": experiment["pr_auc_micro"],
        "pr_auc_macro": experiment["pr_auc_macro"],
        "train_time": experiment["train_time"],
        "inference_time": experiment["inference_time"],
        "vocab_size": experiment["vocab_size"]
    })

comparison_df = pd.DataFrame(best_rows)

comparison_df

,preprocessing,threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro,train_time,inference_time,vocab_size
0,raw,0.25,0.511358,0.364154,0.201227,0.521789,0.394299,78.407964,0.034366,50000
1,repeat,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,83.445783,0.037410,50000
2,repeat_symbols,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,83.782313,0.077956,50000


In [19]:
comparison_df = comparison_df.sort_values(
    by=[
        "micro_f1",
        "macro_f1",
        "hamming_loss"
    ],
    ascending=[
        False,
        False,
        True
    ]
).reset_index(drop=True)

comparison_df

,preprocessing,threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro,train_time,inference_time,vocab_size
0,repeat,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,83.445783,0.037410,50000
1,repeat_symbols,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,83.782313,0.077956,50000
2,raw,0.25,0.511358,0.364154,0.201227,0.521789,0.394299,78.407964,0.034366,50000


In [20]:
BEST_PREPROCESSING = comparison_df.iloc[0]["preprocessing"]
BEST_THRESHOLD = float(comparison_df.iloc[0]["threshold"])

print("Best preprocessing:", BEST_PREPROCESSING)
print("Best threshold:", BEST_THRESHOLD)

Best preprocessing: repeat
Best threshold: 0.25


In [21]:
best_experiment = experiments[BEST_PREPROCESSING]

best_model = best_experiment["model"]
best_vectorizer = best_experiment["vectorizer"]
best_val_prob = best_experiment["y_val_prob"]

best_val_pred = metrics.apply_threshold(
    best_val_prob,
    BEST_THRESHOLD
)

In [22]:
final_val_metrics = metrics.evaluate_multilabel(
    y_val,
    best_val_pred,
    best_val_prob
)

final_val_metrics

{'micro_f1': 0.5122759718017281,
 'macro_f1': 0.36565462491667183,
 'hamming_loss': 0.20063636363636364,
 'pr_auc_macro': np.float64(0.3956354319329758),
 'pr_auc_micro': np.float64(0.5233272362468584)}

In [23]:
label_f1_df = metrics.get_label_wise_f1(
    y_val,
    best_val_pred
)

train_label_counts = y_train.sum(axis=0)

label_f1_df["train_count"] = train_label_counts

label_f1_df = label_f1_df[
    [
        "label_id",
        "label",
        "train_count",
        "f1"
    ]
]

label_f1_df.sort_values(
    "f1",
    ascending=False
)

,label_id,label,train_count,f1
0,0,불평/불만,16696,0.674715
10,10,안타까움/실망,17465,0.656762
23,23,어이없음,15996,0.648189
22,22,짜증,14930,0.634897
6,6,화남/분노,12299,0.610465
28,28,즐거움/신남,10867,0.581201
20,20,한심함,12201,0.579897
42,42,기쁨,9967,0.572024
2,2,감동/감탄,10778,0.568688
12,12,의심/불신,11565,0.547741


In [24]:
label_f1_df.sort_values(
    "f1",
    ascending=True
).head(15)

,label_id,label,train_count,f1
30,30,죄책감,698,0.000000
18,18,공포/무서움,1194,0.000000
17,17,부끄러움,2329,0.013699
25,25,패배/자기혐오,1581,0.031250
36,36,서러움,2031,0.061303
26,26,귀찮음,2428,0.065672
19,19,절망,3652,0.096475
35,35,부담/안_내킴,4768,0.112175
27,27,힘듦/지침,3769,0.162866
11,11,비장함,3352,0.192593


In [25]:
label_f1_df.sort_values(
    "train_count",
    ascending=True
).head(15)

,label_id,label,train_count,f1
30,30,죄책감,698,0.000000
18,18,공포/무서움,1194,0.000000
25,25,패배/자기혐오,1581,0.031250
36,36,서러움,2031,0.061303
17,17,부끄러움,2329,0.013699
26,26,귀찮음,2428,0.065672
11,11,비장함,3352,0.192593
19,19,절망,3652,0.096475
37,37,재미없음,3691,0.400598
27,27,힘듦/지침,3769,0.162866


In [26]:
error_count = 0

for i in range(len(val_df)):

    true_ids = np.where(
        y_val[i] == 1
    )[0]

    pred_ids = np.where(
        best_val_pred[i] == 1
    )[0]

    if set(true_ids) != set(pred_ids):

        print("TEXT:")
        print(val_df.iloc[i]["text"])

        print("TRUE:")
        print([
            utils_labels.LABELS[j]
            for j in true_ids
        ])

        print("PRED:")
        print([
            utils_labels.LABELS[j]
            for j in pred_ids
        ])

        print("-" * 80)

        error_count += 1

        if error_count == 10:
            break

TEXT:
음료가 몇푼이나 한다고.. 미국같으면 바로 바꿔줄텐데 참 어이없네요 그 카페
TRUE:
['불평/불만', '화남/분노', '안타까움/실망', '짜증', '어이없음', '서러움']
PRED:
['불평/불만', '화남/분노', '안타까움/실망', '의심/불신', '한심함', '짜증', '어이없음', '즐거움/신남', '당황/난처']
--------------------------------------------------------------------------------
TEXT:
아버님 끝까지 진실이 밝혀질때까지 응원합니다.
TRUE:
['환영/호의', '감동/감탄', '존경', '기대감', '비장함', '신기함/관심', '아껴주는', '불쌍함/연민', '안심/신뢰']
PRED:
['환영/호의', '감동/감탄', '고마움', '존경', '기대감', '안타까움/실망', '신기함/관심', '아껴주는', '안심/신뢰']
--------------------------------------------------------------------------------
TEXT:
대응이 잘못됐어 차라리 그냥 철없던 시절 그랬었다고 인정하고 울면서 자숙하는 시늉만 했어도 이렇게까지 최악은 아니었을텐데
TRUE:
['불평/불만', '안타까움/실망', '의심/불신', '한심함', '짜증', '어이없음', '증오/혐오', '서러움']
PRED:
['불평/불만', '화남/분노', '안타까움/실망', '의심/불신', '한심함', '짜증', '어이없음', '당황/난처']
--------------------------------------------------------------------------------
TEXT:
저번주 일요일 지리산둘레길엘 갔다왔습니다. "강호동,은지원 코스" 상상이하!!! 기대가 컸던 만큼 실망도 크더군요.
TRUE:
['불평/불만', '안타까움/실망', '짜증']
PRED:
['불평/불만', '안타까움/실망', '의심/불신', '신기함/관심

In [ ]:
comparison_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "week04_lr_preprocessing_comparison.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

label_f1_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "week04_lr_label_f1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import os

ARTIFACT_DIR = "/content/drive/MyDrive/26_KOTE/artifacts/week4_lr"

os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("exists:", os.path.exists(ARTIFACT_DIR))

ARTIFACT_DIR: /content/drive/MyDrive/26_KOTE/artifacts/week4_lr
exists: True


In [ ]:
joblib.dump(
    best_model,
    os.path.join(
        ARTIFACT_DIR,
        "logistic_regression.joblib"
    )
)

joblib.dump(
    best_vectorizer,
    os.path.join(
        ARTIFACT_DIR,
        "tfidf_vectorizer.joblib"
    )
)

joblib.dump(
    mlb,
    os.path.join(
        ARTIFACT_DIR,
        "multilabel_binarizer.joblib"
    )
)

['/content/drive/MyDrive/26_KOTE/artifacts/week4_lr/multilabel_binarizer.joblib']

In [ ]:
best_condition = preprocessing_conditions[
    BEST_PREPROCESSING
]

config = {
    "model": "Logistic Regression",
    "vectorizer": "TF-IDF",

    "preprocessing": BEST_PREPROCESSING,

    "normalize_repeat":
        best_condition["normalize_repeat"],

    "remove_symbols":
        best_condition["remove_symbols"],

    "threshold":
        BEST_THRESHOLD,

    "num_labels":
        NUM_LABELS,

    "tfidf": {
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_features": 50000,
        "sublinear_tf": True
    }
}

with open(
    os.path.join(
        ARTIFACT_DIR,
        "config.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
print(os.listdir(ARTIFACT_DIR))

['logistic_regression.joblib', 'tfidf_vectorizer.joblib', 'multilabel_binarizer.joblib', 'config.json']


In [ ]:
!ls /content

drive  sample_data  Self-Project
